Mathematical Layer EduLink

#  Install PySpark

In [ ]:
!pip install pyspark==3.5.1 -q
print("PySpark installed")

PySpark installed


#  Start Spark Session

In [ ]:
import os
os.environ["PYSPARK_PYTHON"]        = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("EduLink_Mathematical_Layer")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark started successfully")

Spark version: 3.5.1
Spark started successfully


# Load Dataset

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, DoubleType, StringType
)

# Upload student_details.csv to Colab first
INPUT_FILE  = "student_details.csv"
OUTPUT_FILE = "synthetic_exam_master_2000.csv"

df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_FILE))

print("Loaded dataset:")
print("  Rows:   ", df.count())
print("  Columns:", len(df.columns))
df.printSchema()
df.show(3, truncate=False)

Loaded dataset:
  Rows:    2000
  Columns: 61
root
 |-- student_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- school: string (nullable = true)
 |-- district: string (nullable = true)
 |-- level: string (nullable = true)
 |-- stream: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- location: string (nullable = true)
 |-- time_horizon: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- device_access: string (nullable = true)
 |-- english_level: string (nullable = true)
 |-- scholarship_need: string (nullable = true)
 |-- ol_math: string (nullable = true)
 |-- ol_science: string (nullable = true)
 |-- ol_ict: string (nullable = true)
 |-- Q1: integer (nullable = true)
 |-- Q2: integer (nullable = true)
 |-- Q3: integer (nullable = true)
 |-- Q4: integer (nullable = true)
 |-- Q5: integer (nullable = true)
 |-- Q6: integer (nullable = true)
 |--

# Validate and Clean MCQ Columns

In [ ]:
# Ensure all Q1-Q40 are integer and clipped to 1-5
for i in range(1, 41):
    col_name = f"Q{i}"
    df = df.withColumn(
        col_name,
        F.when(F.col(col_name).isNull(), 3)
         .otherwise(
             F.greatest(F.lit(1),
             F.least(F.lit(5),
             F.col(col_name).cast(IntegerType())))
         )
    )

print("All 40 MCQ columns validated and clipped to 1-5")

# Verify range
from pyspark.sql.functions import min as spark_min, max as spark_max
q_cols = [f"Q{i}" for i in range(1, 41)]
mins   = [spark_min(F.col(c)) for c in q_cols]
maxes  = [spark_max(F.col(c)) for c in q_cols]

result = df.agg(*mins, *maxes).collect()[0]
print("Q min across all columns:", min(result[:40]))
print("Q max across all columns:", max(result[40:]))

All 40 MCQ columns validated and clipped to 1-5
Q min across all columns: 1
Q max across all columns: 5


# Define Feature Map

In [ ]:
# Question-to-composite mapping
# Based on Holland RIASEC + Trait-Factor Theory + SCCT
# Each group measures the same latent psychological trait

FEATURE_MAP = {
    "Analytical_Thinking":          [1, 2, 3, 4, 16],
    "Structure_Preference":         [6, 7, 14, 19],
    "Creativity_Index":             [5, 24, 30, 40],
    "Emotional_Stability":          [9],
    "Strategic_Vision":             [8, 20, 32, 39],
    "Innovation_Drive":             [5, 21, 24, 30, 40],
    "Social_Intelligence":          [10, 13, 17, 27],
    "Data_Literacy":                [11, 18, 28],
    "Tech_Adaptability":            [12, 22, 31, 37],
    "Communication_Skill":          [6, 10, 13, 17],
    "Technical_ProblemSolving":     [14, 16, 19, 21, 38],
    "Leadership_Capability":        [15, 32, 35],
    "Process_Optimization":         [14, 19, 20, 38],
    "Tech_Interest":                [22, 30, 31, 34, 40],
    "Business_Economics_Interest":  [23, 25, 26, 34],
    "Social_Impact_Motivation":     [10, 27, 39],
    "Future_Orientation":           [8, 22, 33, 37, 39],
    "Career_Growth_Mindset":        [31, 33, 36, 37, 38],
    "Entrepreneurship_Orientation": [26, 32, 34, 35],
    "Global_Innovation_Alignment":  [31, 37, 39, 40],
}

FEATURES = list(FEATURE_MAP.keys())
print(f"Total composite features defined: {len(FEATURES)}")

Total composite features defined: 20


# Compute 20 Composite Features (PySpark)

In [ ]:
# Formula: composite = average(selected Q answers) / 5
# Output range: 0.0 to 1.0
# Pure PySpark column expressions — no pandas

for feature, questions in FEATURE_MAP.items():
    q_cols    = [F.col(f"Q{q}").cast(DoubleType()) for q in questions]
    n         = len(q_cols)
    total_sum = q_cols[0]
    for c in q_cols[1:]:
        total_sum = total_sum + c
    df = df.withColumn(
        feature,
        (total_sum / F.lit(n) / F.lit(5.0)).cast(DoubleType())
    )

print("Computed all 20 composite features.")
print()

# Verify range
print("Composite feature ranges (should be 0.0 - 1.0):")
for feat in FEATURES:
    mn = df.agg(spark_min(feat)).collect()[0][0]
    mx = df.agg(spark_max(feat)).collect()[0][0]
    print(f"  {feat}: {mn:.3f} - {mx:.3f}")

Computed all 20 composite features.

Composite feature ranges (should be 0.0 - 1.0):
  Analytical_Thinking: 0.360 - 0.960
  Structure_Preference: 0.400 - 1.000
  Creativity_Index: 0.350 - 0.950
  Emotional_Stability: 0.200 - 1.000
  Strategic_Vision: 0.350 - 0.900
  Innovation_Drive: 0.360 - 0.920
  Social_Intelligence: 0.400 - 0.950
  Data_Literacy: 0.333 - 1.000
  Tech_Adaptability: 0.350 - 0.950
  Communication_Skill: 0.450 - 1.000
  Technical_ProblemSolving: 0.400 - 0.960
  Leadership_Capability: 0.200 - 1.000
  Process_Optimization: 0.350 - 1.000
  Tech_Interest: 0.360 - 0.920
  Business_Economics_Interest: 0.400 - 0.950
  Social_Impact_Motivation: 0.333 - 1.000
  Future_Orientation: 0.400 - 0.920
  Career_Growth_Mindset: 0.360 - 0.880
  Entrepreneurship_Orientation: 0.350 - 0.900
  Global_Innovation_Alignment: 0.350 - 1.000


# Compute 6 RIASEC Scores (PySpark)

In [ ]:
# RIASEC weighted aggregation from composite features
# Formula: RIASEC = sum(feature * weight) / sum(weights) * 100
# Based on Holland theory + O*NET importance ratings

RIASEC_WEIGHTS = {
    "R": {
        "Technical_ProblemSolving": 1.0,
        "Process_Optimization":     0.6,
        "Tech_Adaptability":        0.4,
    },
    "I": {
        "Analytical_Thinking":      1.0,
        "Data_Literacy":            0.9,
        "Strategic_Vision":         0.5,
        "Tech_Interest":            0.4,
    },
    "A": {
        "Creativity_Index":              1.0,
        "Innovation_Drive":              0.8,
        "Global_Innovation_Alignment":   0.4,
    },
    "S": {
        "Social_Intelligence":      1.0,
        "Communication_Skill":      0.9,
        "Social_Impact_Motivation": 0.6,
    },
    "E": {
        "Leadership_Capability":         1.0,
        "Entrepreneurship_Orientation":  0.9,
        "Business_Economics_Interest":   0.7,
        "Career_Growth_Mindset":         0.4,
    },
    "C": {
        "Structure_Preference":  1.0,
        "Process_Optimization":  0.8,
        "Data_Literacy":         0.5,
        "Emotional_Stability":   0.3,
    },
}

for letter, weights in RIASEC_WEIGHTS.items():
    total_w    = sum(weights.values())
    items      = list(weights.items())
    weighted   = F.col(items[0][0]) * F.lit(items[0][1])
    for feat, w in items[1:]:
        weighted = weighted + F.col(feat) * F.lit(w)
    df = df.withColumn(
        letter,
        (weighted / F.lit(total_w) * F.lit(100.0)).cast(DoubleType())
    )

print("Computed 6 RIASEC scores.")
print()
print("RIASEC score ranges (should be 0-100):")
for letter in ["R","I","A","S","E","C"]:
    mn = df.agg(spark_min(letter)).collect()[0][0]
    mx = df.agg(spark_max(letter)).collect()[0][0]
    print(f"  {letter}: {mn:.1f} - {mx:.1f}")

Computed 6 RIASEC scores.

RIASEC score ranges (should be 0-100):
  R: 45.5 - 91.5
  I: 52.4 - 84.4
  A: 40.4 - 91.5
  S: 45.4 - 93.2
  E: 43.3 - 83.8
  C: 50.5 - 92.3


# Generate Interest Code (PySpark UDF)

In [ ]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

# UDF: sort RIASEC scores and return top 3 letters
def get_interest_code(R, I, A, S, E, C):
    scores  = {"R":R, "I":I, "A":A, "S":S, "E":E, "C":C}
    sorted_ = sorted(scores, key=scores.get, reverse=True)
    return "".join(sorted_[:3])

interest_code_udf = udf(get_interest_code, StringType())

df = df.withColumn(
    "interest_code",
    interest_code_udf(
        F.col("R"), F.col("I"), F.col("A"),
        F.col("S"), F.col("E"), F.col("C")
    )
)

print("Interest code distribution (top 15):")
df.groupBy("interest_code").count()\
  .orderBy(F.col("count").desc())\
  .show(15)

print("Total unique codes:", df.select("interest_code").distinct().count())

Interest code distribution (top 15):
+-------------+-----+
|interest_code|count|
+-------------+-----+
|          SCR|   59|
|          SAI|   55|
|          SIC|   52|
|          RCI|   45|
|          ASI|   44|
|          SRC|   40|
|          RIC|   38|
|          CRI|   37|
|          SEI|   34|
|          AIR|   34|
|          SIA|   34|
|          ISC|   33|
|          SIR|   33|
|          CSR|   30|
|          CIR|   30|
+-------------+-----+
only showing top 15 rows

Total unique codes: 119


# Assign Career Cluster (PySpark UDF with Softmax)

In [ ]:
import hashlib
import math
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

def assign_cluster(
    Analytical_Thinking, Structure_Preference, Creativity_Index,
    Emotional_Stability, Strategic_Vision, Innovation_Drive,
    Social_Intelligence, Data_Literacy, Tech_Adaptability,
    Communication_Skill, Technical_ProblemSolving, Leadership_Capability,
    Process_Optimization, Tech_Interest, Business_Economics_Interest,
    Social_Impact_Motivation, Future_Orientation, Career_Growth_Mindset,
    Entrepreneurship_Orientation, Global_Innovation_Alignment,
    R, I, A, S, E, C
):
    # Score each cluster using weighted composites + RIASEC
    scores = {
        "Data_AI_Engineering":
            1.25*Analytical_Thinking +
            1.25*Data_Literacy +
            0.90*Tech_Interest +
            0.35*Tech_Adaptability +
            0.25*I/100,

        "Software_Web_Engineering":
            1.15*Tech_Adaptability +
            1.05*Technical_ProblemSolving +
            0.75*Analytical_Thinking +
            0.65*Tech_Interest +
            0.25*R/100,

        "Network_Infrastructure":
            1.20*Technical_ProblemSolving +
            1.10*Process_Optimization +
            1.00*Structure_Preference +
            0.55*Tech_Adaptability +
            0.25*R/100,

        "IT_Operations_QA":
            1.20*Structure_Preference +
            1.05*Process_Optimization +
            0.85*Analytical_Thinking +
            0.65*Tech_Adaptability +
            0.25*C/100,

        "UX_Creative_Tech":
            1.35*Creativity_Index +
            0.90*Innovation_Drive +
            0.80*Social_Intelligence +
            0.75*Communication_Skill +
            0.25*A/100,

        "Business_IT_Management":
            1.25*Leadership_Capability +
            1.05*Business_Economics_Interest +
            0.85*Communication_Skill +
            0.60*Social_Intelligence +
            0.25*E/100,

        "Digital_Marketing_Media":
            1.25*Communication_Skill +
            1.05*Social_Intelligence +
            0.95*Entrepreneurship_Orientation +
            0.80*Creativity_Index +
            0.25*A/100,

        "Hardware_Systems":
            1.25*Technical_ProblemSolving +
            0.85*Analytical_Thinking +
            0.80*Tech_Adaptability +
            0.60*Process_Optimization +
            0.25*R/100,
    }

    # Deterministic argmax — always picks highest scoring cluster
    # Same student always gets same cluster
    # No randomness — ML model can learn this pattern properly
    best_cluster = None
    best_score   = -1.0

    for cluster, score in scores.items():
        if score > best_score:
            best_score   = score
            best_cluster = cluster

    return best_cluster


assign_cluster_udf = udf(assign_cluster, StringType())

FEATURE_COLS = [
    "Analytical_Thinking", "Structure_Preference", "Creativity_Index",
    "Emotional_Stability", "Strategic_Vision", "Innovation_Drive",
    "Social_Intelligence", "Data_Literacy", "Tech_Adaptability",
    "Communication_Skill", "Technical_ProblemSolving", "Leadership_Capability",
    "Process_Optimization", "Tech_Interest", "Business_Economics_Interest",
    "Social_Impact_Motivation", "Future_Orientation", "Career_Growth_Mindset",
    "Entrepreneurship_Orientation", "Global_Innovation_Alignment",
    "R", "I", "A", "S", "E", "C"
]

df = df.withColumn(
    "career_cluster",
    assign_cluster_udf(*[F.col(c) for c in FEATURE_COLS])
)

print("Career cluster distribution:")
df.groupBy("career_cluster").count()\
  .orderBy(F.col("count").desc())\
  .show()

total_clusters = df.select("career_cluster").distinct().count()
print("Unique clusters:", total_clusters)
print("All 8 clusters covered:", total_clusters == 8)

Career cluster distribution:
+--------------------+-----+
|      career_cluster|count|
+--------------------+-----+
|Digital_Marketing...|  891|
|Network_Infrastru...|  455|
| Data_AI_Engineering|  350|
|    UX_Creative_Tech|  123|
|    IT_Operations_QA|  106|
|Business_IT_Manag...|   64|
|Software_Web_Engi...|   11|
+--------------------+-----+

Unique clusters: 7
All 8 clusters covered: False


# Compute Section Scores (PySpark)

In [ ]:
# Section-level aggregate scores for explainability
# Each section covers 10 questions

def section_avg(q_range):
    cols  = [F.col(f"Q{i}").cast(DoubleType()) for i in q_range]
    total = cols[0]
    for c in cols[1:]:
        total = total + c
    return (total / F.lit(len(cols)) / F.lit(5.0)).cast(DoubleType())

df = df.withColumn(
    "Section1_Personality_Cognitive",
    section_avg(range(1, 11))
)
df = df.withColumn(
    "Section2_Skills_Capability",
    section_avg(range(11, 21))
)
df = df.withColumn(
    "Section3_Interests_Motivation",
    section_avg(range(21, 31))
)
df = df.withColumn(
    "Section4_Career_Orientation",
    section_avg(range(31, 41))
)

print("Section scores computed.")
df.select(
    "Section1_Personality_Cognitive",
    "Section2_Skills_Capability",
    "Section3_Interests_Motivation",
    "Section4_Career_Orientation"
).show(5)

Section scores computed.
+------------------------------+--------------------------+-----------------------------+---------------------------+
|Section1_Personality_Cognitive|Section2_Skills_Capability|Section3_Interests_Motivation|Section4_Career_Orientation|
+------------------------------+--------------------------+-----------------------------+---------------------------+
|                          0.74|                       0.7|                         0.64|                       0.58|
|                          0.64|                      0.76|                         0.78|                       0.62|
|                          0.72|                      0.72|                         0.72|                       0.62|
|                          0.72|                      0.62|                         0.62|                       0.62|
|                          0.58|                      0.64|                         0.76|                       0.72|
+------------------------------

# Compute Writing Metrics (PySpark UDF)

In [ ]:
import re as re_module
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType
from pyspark.sql.functions import udf

writing_schema = StructType([
    StructField("writing_word_count",       IntegerType(), True),
    StructField("writing_sentence_count",   IntegerType(), True),
    StructField("writing_avg_word_length",  DoubleType(),  True),
    StructField("writing_length_score",     DoubleType(),  True),
    StructField("writing_structure_score",  DoubleType(),  True),
    StructField("writing_vocabulary_score", DoubleType(),  True),
    StructField("writing_clarity_score",    DoubleType(),  True),
    StructField("writing_overall_score",    DoubleType(),  True),
    StructField("writing_level",            StringType(),  True),
])

def compute_writing_metrics(text):
    if text is None:
        text = ""
    text      = str(text)
    words     = re_module.findall(r"\b[a-zA-Z]+\b", text)
    sentences = [s for s in re_module.split(r"[.!?]+", text) if s.strip()]

    word_count     = len(words)
    sentence_count = max(len(sentences), 1)
    avg_word_len   = (sum(len(w) for w in words) / word_count
                      if word_count > 0 else 0.0)

    length_score     = min(word_count / 150.0 * 100, 100.0)
    structure_score  = min(sentence_count / 6.0 * 100, 100.0)
    vocab_score      = min(avg_word_len / 6.0 * 100, 100.0)
    clarity_score    = 100.0 - min(abs(word_count - 125) / 125.0 * 60, 60.0)
    overall          = (length_score + structure_score +
                        vocab_score + clarity_score) / 4.0

    level = ("Strong" if overall >= 80
             else "Moderate" if overall >= 60
             else "Needs_Improvement")

    return (
        word_count, sentence_count,
        round(avg_word_len, 2),
        round(length_score, 2),
        round(structure_score, 2),
        round(vocab_score, 2),
        round(clarity_score, 2),
        round(overall, 2),
        level
    )

writing_udf = udf(compute_writing_metrics, writing_schema)

if "writing_sample" in df.columns:
    df = df.withColumn(
        "writing_metrics",
        writing_udf(F.col("writing_sample"))
    )
    for field in writing_schema.fields:
        df = df.withColumn(
            field.name,
            F.col(f"writing_metrics.{field.name}")
        )
    df = df.drop("writing_metrics")
    print("Writing metrics computed.")
    df.select([f.name for f in writing_schema.fields]).show(5)
else:
    print("No writing_sample column — skipping.")

Writing metrics computed.
+------------------+----------------------+-----------------------+--------------------+-----------------------+------------------------+---------------------+---------------------+-----------------+
|writing_word_count|writing_sentence_count|writing_avg_word_length|writing_length_score|writing_structure_score|writing_vocabulary_score|writing_clarity_score|writing_overall_score|    writing_level|
+------------------+----------------------+-----------------------+--------------------+-----------------------+------------------------+---------------------+---------------------+-----------------+
|                23|                     1|                   4.43|               15.33|                  16.67|                   73.91|                51.04|                39.24|Needs_Improvement|
|                28|                     2|                   4.11|               18.67|                  33.33|                   68.45|                53.44|               

In [ ]:
# Save writing_samples.csv for Writing Analysis model
writing_samples = df.select(
    F.col("student_id"),
    F.col("writing_sample").alias("writing_text")
)
writing_samples.coalesce(1)\
    .write.option("header", True)\
    .mode("overwrite")\
    .csv("writing_samples_temp")

import glob, shutil
parts = glob.glob("writing_samples_temp/part-*.csv")
if parts:
    shutil.copy(parts[0], "writing_samples.csv")
    shutil.rmtree("writing_samples_temp")

print("Saved: writing_samples.csv for Writing Analysis model")

Saved: writing_samples.csv for Writing Analysis model


# Save Output Dataset

In [ ]:
# Drop columns not needed in output
DROP_COLS = [
    "writing_sample", "writing_prompt",
    "form_version", "consent_participation",
    "anonymous_data_consent"
]

output_df = df
for col in DROP_COLS:
    if col in output_df.columns:
        output_df = output_df.drop(col)

# Save as single CSV file
(output_df
    .coalesce(1)
    .write
    .option("header", True)
    .mode("overwrite")
    .csv("output_temp"))

# Rename the part file to proper name
import glob, shutil, os
part_files = glob.glob("output_temp/part-*.csv")
if part_files:
    shutil.copy(part_files[0], OUTPUT_FILE)
    shutil.rmtree("output_temp")

print("="*50)
print("SAVED:", OUTPUT_FILE)
print("Rows:", output_df.count())
print("Columns:", len(output_df.columns))
print()
print("career_cluster distribution:")
output_df.groupBy("career_cluster").count()\
    .orderBy(F.col("count").desc()).show()

SAVED: synthetic_exam_master_2000.csv
Rows: 2000
Columns: 99

career_cluster distribution:
+--------------------+-----+
|      career_cluster|count|
+--------------------+-----+
|Digital_Marketing...|  891|
|Network_Infrastru...|  455|
| Data_AI_Engineering|  350|
|    UX_Creative_Tech|  123|
|    IT_Operations_QA|  106|
|Business_IT_Manag...|   64|
|Software_Web_Engi...|   11|
+--------------------+-----+



# Validation Report

In [ ]:
print("=== VALIDATION REPORT ===")
print()

total = output_df.count()
print(f"1. Total students:        {total}")
print(f"2. Missing values check:")

for col_name in FEATURES + ["R","I","A","S","E","C","interest_code","career_cluster"]:
    null_count = output_df.filter(F.col(col_name).isNull()).count()
    status     = "OK" if null_count == 0 else f"ERROR ({null_count} nulls)"
    print(f"   {col_name}: {status}")

print()
print(f"3. Unique clusters: {output_df.select('career_cluster').distinct().count()}")
print()
print("4. Cluster distribution:")
output_df.groupBy("career_cluster").count()\
    .orderBy(F.col("count").desc()).show()

print()
print("5. RIASEC score ranges:")
from pyspark.sql.functions import min as smin, max as smax, round as sround
output_df.agg(
    *[sround(smin(r), 1).alias(f"{r}_min") for r in "RIASEC"],
    *[sround(smax(r), 1).alias(f"{r}_max") for r in "RIASEC"]
).show()

print("=== READY FOR CAREER FIT MODEL ===")

=== VALIDATION REPORT ===

1. Total students:        2000
2. Missing values check:
   Analytical_Thinking: OK
   Structure_Preference: OK
   Creativity_Index: OK
   Emotional_Stability: OK
   Strategic_Vision: OK
   Innovation_Drive: OK
   Social_Intelligence: OK
   Data_Literacy: OK
   Tech_Adaptability: OK
   Communication_Skill: OK
   Technical_ProblemSolving: OK
   Leadership_Capability: OK
   Process_Optimization: OK
   Tech_Interest: OK
   Business_Economics_Interest: OK
   Social_Impact_Motivation: OK
   Future_Orientation: OK
   Career_Growth_Mindset: OK
   Entrepreneurship_Orientation: OK
   Global_Innovation_Alignment: OK
   R: OK
   I: OK
   A: OK
   S: OK
   E: OK
   C: OK
   interest_code: OK
   career_cluster: OK

3. Unique clusters: 8

4. Cluster distribution:
+--------------------+-----+
|      career_cluster|count|
+--------------------+-----+
|Digital_Marketing...|  581|
|Network_Infrastru...|  366|
| Data_AI_Engineering|  284|
|    IT_Operations_QA|  215|
|    UX_Cre